In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import Dense

In [ ]:
df = pd.read_csv('data/power_consumption_2017.csv')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

df['DateTime'] = pd.to_datetime(df['DateTime'])
print(df.head())

# 시간 정보 뺀 데이터 섞기 전 인덱스 저장
#original_indices = df.index

#df.set_index('DateTime', inplace=True)
print(df.shape)
print(df.columns)

In [ ]:
df

In [ ]:
missing_values = df[['Zone 1 Power Consumption','Zone 2  Power Consumption', 'Zone 3  Power Consumption']].isnull().sum()
missing_values

**시간에 따른 발전량**

In [ ]:
# 'Zone 1 Power Consumption' 열을 시각화
plt.figure(figsize=(12, 6))
plt.plot(df['Zone 1 Power Consumption'], label='Zone 1', linestyle='-', marker='o', color='skyblue', linewidth=0.5, markersize=3)
plt.plot(df['Zone 2  Power Consumption'], label='Zone 2', linestyle='-', marker='o', color='orange', linewidth=0.5, markersize=3)
plt.plot(df['Zone 3  Power Consumption'], label='Zone 3', linestyle='-', marker='o', color='green', linewidth=0.5, markersize=3)
plt.title('Actual Power Consumption Over Time')
plt.xlabel('Time')
plt.ylabel('Power Consumption')
plt.legend()
plt.show()

**온도에 따른 발전량**

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['Temperature'], df['Zone 1 Power Consumption'], label='Zone 1', color='blue', alpha=0.7)
plt.scatter(df['Temperature'], df['Zone 2  Power Consumption'], label='Zone 2', color='orange', alpha=0.7)
plt.scatter(df['Temperature'], df['Zone 3  Power Consumption'], label='Zone 3', color='green', alpha=0.7)
plt.title('Power Consumption vs. Temperature by Zone')
plt.xlabel('Temperature')
plt.ylabel('Power Consumption')
plt.legend()
plt.show()

In [ ]:
correlation_zone1 = df['Temperature'].corr(df['Zone 1 Power Consumption'])
correlation_zone2 = df['Temperature'].corr(df['Zone 2  Power Consumption'])
correlation_zone3 = df['Temperature'].corr(df['Zone 3  Power Consumption'])
print(f"{correlation_zone1:.4f}, {correlation_zone2:.4f}, {correlation_zone3:.4f}")

=> Zone3이 온도에 따른 발전량의 상관계수가 가장 높음

**습도에 따른 발전량**

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['Humidity'], df['Zone 1 Power Consumption'], label='Zone 1', color='blue', alpha=0.7)
plt.scatter(df['Humidity'], df['Zone 2  Power Consumption'], label='Zone 2', color='orange', alpha=0.7)
plt.scatter(df['Humidity'], df['Zone 3  Power Consumption'], label='Zone 3', color='green', alpha=0.7)
plt.title('Power Consumption vs. Humidity by Zone')
plt.xlabel('Humidity')
plt.ylabel('Power Consumption')
plt.legend()
plt.show()

In [ ]:
correlation_zone1 = df['Humidity'].corr(df['Zone 1 Power Consumption'])
correlation_zone2 = df['Humidity'].corr(df['Zone 2  Power Consumption'])
correlation_zone3 = df['Humidity'].corr(df['Zone 3  Power Consumption'])
print(f"{correlation_zone1:.4f}, {correlation_zone2:.4f}, {correlation_zone3:.4f}")

=> 습도에 따른 발전량의 상관계수 낮음

**풍속에 따른 발전량**

In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(df['Wind Speed'], df['Zone 1 Power Consumption'], label='Zone 1', color='blue', alpha=0.7)
plt.scatter(df['Wind Speed'], df['Zone 2  Power Consumption'], label='Zone 2', color='orange', alpha=0.7)
plt.scatter(df['Wind Speed'], df['Zone 3  Power Consumption'], label='Zone 3', color='green', alpha=0.7)
plt.title('Power Consumption vs. Wind Speed by Zone')
plt.xlabel('Wind Speed')
plt.ylabel('Power Consumption')
plt.legend()
plt.show()

In [ ]:
correlation_zone1 = df['Wind Speed'].corr(df['Zone 1 Power Consumption'])
correlation_zone2 = df['Wind Speed'].corr(df['Zone 2  Power Consumption'])
correlation_zone3 = df['Wind Speed'].corr(df['Zone 3  Power Consumption'])
print(f"{correlation_zone1:.4f}, {correlation_zone2:.4f}, {correlation_zone3:.4f}")

=> 바람이 일정할 때엔 발전량이 꾸준히 낮거나 꾸준히 높게 유지가 된다(??)

=> Zone3이 풍속에 따른 발전량의 상관계수가 가장 높지만 유의미한 상관성 없어보임

In [ ]:
correlation_matrix = df.corr()
correlation_matrix

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 모든 변수들간의 상관관계
sns.pairplot(df, kind="scatter")
plt.show()

In [ ]:
## 히트맵
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

# **ANN-2017**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# 회귀 문제니까 정확도(accuracy) 사용X

In [ ]:
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from scipy.stats import loguniform, randint

# Zone 1 Power Consumption

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

In [ ]:
shuffled_df = df.sample(frac=1, random_state=42)     #시간 정보를 기반으로 데이터 섞기
df_sorted = shuffled_df.sort_index()                 #시간 정보를 기반으로 데이터 정렬

In [ ]:
## 독립적인 스케일러 -> 역정규화 수행할 때 필요(훈련 데이터와 테스트 데이터 각각에 대해 독릭접으로 정규화 및 역정규화 수행)
## 하나의 스케일러로 하게되면, 한 데이터셋의 스케일 조정이 다른 데이터셋의 특성에 영향 미칠 수 있고 역정규화 어려울 수 있음

In [ ]:
features = ['Temperature', 'Humidity', 'Wind Speed']
target = ['Zone 1 Power Consumption']

X = shuffled_df[features]
y = shuffled_df[target]


# 데이터 정규화
scaler = MinMaxScaler(feature_range = (0,1))
scaler1 = MinMaxScaler(feature_range = (0,1))
X_scaled = scaler.fit_transform(X)
y_scaled = scaler1.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
train_mean, train_std = np.mean(y_train), np.std(y_train)

In [ ]:
print(f"X_train, X_test, y_train, y_test 개수 : {len(X_train)}, {len(X_test)}, {len(y_train)}, {len(y_test)}")

In [ ]:
len(df)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
def create_model():
  model = Sequential()
  model.add(Dense(256, input_dim=X_train.shape[1], activation='relu'))
  model.add(Dropout(0.2))
  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.2))
  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.2))  # 해당 층 뉴런 중 20%를 무작위로 비활성화
  model.add(Dense(1, activation='linear'))  # 1 : 예측값이 연속적인 경우

  model.compile(optimizer=Adam(learning_rate=0.0001), loss='mean_squared_error', metrics=['mae'])
  return model

# k-fold 교차 검증
k_fold = KFold(n_splits=2, shuffle=True, random_state=42)

mae_scores = []
mse_scores = []

epoch = 70

for fold, (train_idx, valid_idx) in enumerate(k_fold.split(X_train)):
    X_fold_train, X_fold_valid = X_train[train_idx], X_train[valid_idx]
    y_fold_train, y_fold_valid = y_train[train_idx], y_train[valid_idx]

    model = create_model()

    model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=32, verbose=1)

    # 검증 데이터에 대한 예측
    y_fold_valid_pred = model.predict(X_fold_valid)

    mae = mean_absolute_error(y_fold_valid, y_fold_valid_pred)
    mse = mean_squared_error(y_fold_valid, y_fold_valid_pred)
    print(f'Fold {fold + 1}: MAE={mae}, MSE={mse}')
    mae_scores.append(mae)
    mse_scores.append(mse)

In [ ]:
model.summary()

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(mae_scores) + 1)

plt.plot(epochs, mae_scores, label='MAE')
plt.plot(epochs, mse_scores, label='MSE')

plt.xlabel('Fold')
plt.ylabel('Score')
plt.legend()
plt.show()

In [ ]:
# 교차 검증 성능 출력
print(f'Mean MAE across folds: {np.mean(mae_scores)}')
print(f'Mean MSE across folds: {np.mean(mse_scores)}')

**교차검증 후 가장 좋은 모델로 학습 수행**

In [ ]:
best_fold = mae_scores.index(min(mae_scores))
print(f"Best Fold: {best_fold + 1}")

In [ ]:
best_model = create_model()                                             # 최적의 폴드에서 학습한 모델을 새로운 모델로 초기화, 이 모델 이어서 전체 데이터에 대해 추가 학습시키는 용도
best_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)  # 전체 훈련 데이터로 학습

In [ ]:
y_test_pred = best_model.predict(X_test)

In [ ]:
print(f"정규화된 예측값 : {y_test_pred}")   ## 순서대로 아닌 섞인 순서 -> 전체 데이터 아님.
print(f"정규화된 실제값 :{y_test}")

In [ ]:
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f'MAE on Test Data: {mae_test}')
print(f'MSE on Test Data: {mse_test}')

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_test_pred)
print(f'R2 Score: {r2}')

Q . R2 = 0~1 : 1에 가까울수록 모델이 데이터 잘 설명

 R2가 낮더라도 MAE와 MSE가 낮다면 모델이 일부 측면에서는 좋은 예측?

 RNN까지 해보고. . .=> 왜냐면, 데이터 자체가 시간과 관련이 있는 데이터인데 ann으로 하다보니 적절하지 않을 수 있기 때문.

In [ ]:
df

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.scatter(range(len(y_test)), y_test, label='Actual', color='blue')
plt.scatter(range(len(y_test)), y_test_pred, label='Predicted', color='orange')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Values(Scale)')
plt.legend()
plt.show()

In [ ]:
## -------------------------------여기까지 정규화된 값으로 예측값 나옴-----------------------------

### **역 정규화 통해 실제 예측 값 저장하기**

-> 순서대로는 아님

In [ ]:
# 전체 데이터에 대한 예측
y_pred_all = model.predict(X_scaled)

In [ ]:
# 예측 결과를 역정규화 : 순서는 섞여있음
y_pred_rescale = scaler1.inverse_transform(y_pred_all)

# 실제값을 역정규화
y_rescale = scaler1.inverse_transform(y_scaled)

print(f"역정규화된 예측 결과 : {y_pred_rescale}")
print(f"역정규화된 실제값 : {y_rescale}")                 # 실제값을 정규화하고 역정규화해도 동일 값이 나오지 않음(근접할수록 모델이 제대로 예측했다고 볼 수 있음)

In [ ]:
print(len(y_pred_rescale))
print(len(y_rescale))

### **순서대로 정렬**

In [ ]:
#print(max(original_indices))
#print(len(df_sorted))

In [ ]:
array_size = len(y_pred_rescale)
int_indices = df_sorted.index.astype(int) % array_size

In [ ]:
y_pred_rescale_sorted = y_pred_rescale[int_indices]
y_rescale_sorted = y_rescale[int_indices]

print(f"역정규화된 예측 결과(정) : {y_pred_rescale_sorted}")
print(f"역정규화된 실제값(정렬) : {y_rescale_sorted}")          ### 역정규화한거라 실제랑 조금의 차이 존재 !

In [ ]:
result_df = pd.DataFrame({'Predicted_Zone 1' : y_pred_rescale_sorted.flatten()})
df['Predicted_Zone 1'] = result_df['Predicted_Zone 1']                            # 기존 df에 예측한 거 추가

# 수정된 DataFrame을 CSV 파일로 저장
df.to_csv('2017ann_pred_powercomp.csv', index=False)

In [ ]:
plt.scatter(range(len(y_pred_rescale_sorted)), y_rescale_sorted, label='Predicted', color='orange')
plt.scatter(range(len(y_pred_rescale_sorted)), y_pred_rescale_sorted, label='Actual', color='blue')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Values(rescale)')
plt.legend()
plt.show()

# **2018년 예측**

# **0) ann-2018(2018로 학습)**

In [ ]:
df_2018_ann = pd.read_csv('data/power_consumption_2018.csv')
df_2018_ann.fillna(df_2018_ann.mean(), inplace=True)

In [ ]:
shuffled_df = df_2018_ann.sample(frac=1, random_state=42)     #시간 정보를 기반으로 데이터 섞기
df_sorted = shuffled_df.sort_index()                          #시간 정보를 기반으로 데이터 정렬

In [ ]:
feature = ['Temperature','Humidity','Humidity']
target = ['Power Consumption(kWh)']

X = shuffled_df[features]
y = shuffled_df[target]

scaler = MinMaxScaler(feature_range = (0,1))
scaler1 = MinMaxScaler(feature_range = (0,1))
X_scaled = scaler.fit_transform(X)
y_scaled = scaler1.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
train_mean, train_std = np.mean(y_train), np.std(y_train)

In [ ]:
print(f"X_train, X_test, y_train, y_test 개수 : {len(X_train)}, {len(X_test)}, {len(y_train)}, {len(y_test)}")

In [ ]:
def create_model():
  model = Sequential()
  model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
  model.add(Dropout(0.2))
  model.add(Dense(64, activation='relu'))
  model.add(Dropout(0.2))
  model.add(Dense(32, activation='relu'))
  model.add(Dropout(0.2))
  model.add(Dense(1, activation='linear'))

  model.compile(optimizer=Adam(learning_rate=0.0001), loss='mean_squared_error', metrics=['mae'])
  return model

# k-fold 교차 검증
k_fold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
mse_scores = []

epoch = 50

for fold, (train_idx, valid_idx) in enumerate(k_fold.split(X_train)):
    X_fold_train, X_fold_valid = X_train[train_idx], X_train[valid_idx]
    y_fold_train, y_fold_valid = y_train[train_idx], y_train[valid_idx]

    model = create_model()

    model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=32, verbose=1)

    # 검증 데이터에 대한 예측
    y_fold_valid_pred = model.predict(X_fold_valid)

    mae = mean_absolute_error(y_fold_valid, y_fold_valid_pred)
    mse = mean_squared_error(y_fold_valid, y_fold_valid_pred)
    print(f'Fold {fold + 1}: MAE={mae}, MSE={mse}')
    mae_scores.append(mae)
    mse_scores.append(mse)

In [ ]:
print(f'Mean MAE across folds: {np.mean(mae_scores)}')
print(f'Mean MSE across folds: {np.mean(mse_scores)}')

In [ ]:
best_fold = mae_scores.index(min(mae_scores))
print(f"Best Fold: {best_fold + 1}")

In [ ]:
best_model = create_model()
best_model.fit(X_train, y_train, epochs=100, batch_size=32, verbose=1)

In [ ]:
y_test_pred = best_model.predict(X_test)

In [ ]:
print(f"정규화된 예측값 : {y_test_pred}")   ## 순서대로 아닌 섞인 순서 -> 전체 데이터 아님.
print(f"정규화된 실제값 :{y_test}")

In [ ]:
mae_test = mean_absolute_error(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)

print(f'MAE on Test Data: {mae_test}')
print(f'MSE on Test Data: {mse_test}')

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_test, y_test_pred)
print(f'R2 Score: {r2}')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.scatter(range(len(y_test)), y_test, label='Actual', color='skyblue')
plt.scatter(range(len(y_test)), y_test_pred, label='Predicted', color='purple')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('2018-Actual vs Predicted Values(Scale)')
plt.legend()
plt.show()

In [ ]:
y_pred_all = model.predict(X_scaled)                     #전체 데이터 예측

# 예측 결과를 역정규화 : 순서는 섞여있음
y_pred_rescale = scaler1.inverse_transform(y_pred_all)

# 실제값을 역정규화
y_rescale = scaler1.inverse_transform(y_scaled)

print(f"역정규화된 예측 결과 : {y_pred_rescale}")
print(f"역정규화된 실제값 : {y_rescale}")

In [ ]:
# 정렬 : 정렬이 왜 안되지..
array_size = len(y_pred_rescale)
int_indices = df_sorted.index.astype(int) % array_size

y_pred_rescale_sorted = y_pred_rescale[int_indices]
y_rescale_sorted = y_rescale[int_indices]

print(f"역정규화된 예측 결과(정렬) : {y_pred_rescale_sorted}")
print(f"역정규화된 실제값(정렬) : {y_rescale_sorted}")

In [ ]:
# result_df = pd.DataFrame({'Pred_Power Consumption(kWh)' : y_pred_rescale2.flatten()})
# df2['Pred_Power Consumption(kWh)'] = result_df['Pred_Power Consumption(kWh)']                            # 기존 df2에 예측한 거 추가

# df.to_csv('2018ann_pred_충북 충주 201801-201812.csv', index=False)

In [ ]:
plt.scatter(range(len(y_rescale_sorted)), y_rescale_sorted, label='Actual', color='skyblue')
plt.scatter(range(len(y_rescale_sorted)), y_pred_rescale_sorted, label='Predicted', color='purple')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('2018-Actual vs Predicted Values(rescale)')
plt.legend()
plt.show()

# **1) ANN-2018 (2017모델 사용)**

In [ ]:
df2 = pd.read_csv('data/power_consumption_2018.csv')
df2.fillna(df2.mean(), inplace=True)

feature = ['Temperature','Humidity','Humidity']
target = ['Power Consumption(kWh)']

X1 = df2[feature]
y_test1 = df2[target]

scaler = MinMaxScaler(feature_range = (0,1))
scaler1 = MinMaxScaler(feature_range = (0,1))
X_scaled = scaler.fit_transform(X1)
y_scaled = scaler1.fit_transform(y_test1)

In [ ]:
y_test_pred = best_model.predict(X_scaled)

In [ ]:
print(f"정규화된 예측값 : {y_test_pred}")   ## 얜 순서대로
print(f"정규화된 실제값 :{y_scaled}")

In [ ]:
mae_test = mean_absolute_error(y_scaled, y_test_pred)
mse_test = mean_squared_error(y_scaled, y_test_pred)

print(f'MAE on Test Data: {mae_test}')
print(f'MSE on Test Data: {mse_test}')

In [ ]:
from sklearn.metrics import r2_score

r2 = r2_score(y_scaled, y_test_pred)
print(f'R2 Score: {r2}')

**역정규화**

In [ ]:
# 전체 데이터에 대한 예측
#y_pred_all = model.predict(X_scaled) #위에서 2018년 데이터로 재정의 함  ##### -> 위에서 전체데이터 했는데 또 할 이유 x

In [ ]:
# 예측 결과를 역정규화 : 순서 그대
y_pred_rescale2 = scaler1.inverse_transform(y_test_pred)

# 실제값을 역정규화
y_rescale2 = scaler1.inverse_transform(y_scaled)

print(f"역정규화된 예측 결과 : {y_pred_rescale2}")
print(f"역정규화된 실제값 : {y_rescale2}")

In [ ]:
result_df = pd.DataFrame({'Pred_Power Consumption(kWh)' : y_pred_rescale2.flatten()})
df2['Pred_Power Consumption(kWh)'] = result_df['Pred_Power Consumption(kWh)']                            # 기존 df2에 예측한 거 추가

df2.to_csv('2018ann_pred_충북 충주 201801-201812.csv', index=False)

In [ ]:
print(len(y_pred_rescale2))
print(len(y_rescale2))

In [ ]:
plt.scatter(range(len(y_rescale2)), y_rescale2, label='Actual', color='blue')
plt.scatter(range(len(y_rescale2)), y_pred_rescale2, label='Predicted', color='orange')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Values(rescale)')
plt.legend()
plt.show()

# **2) RNN-2018**

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

df3 = pd.read_csv("data/power_consumption_2018.csv", usecols=[5], skipfooter=3)
df3.fillna(df.mean(), inplace=True)

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(df3)

def create_dataset(dataset, look_back):
    x_data, y_data = [], []
    for i in range(len(dataset) - look_back - 1):
        data = dataset[i:(i + look_back), 0]
        x_data.append(data)
        y_data.append(dataset[i + look_back, 0])
    return np.array(x_data), np.array(y_data)

look_back = 3
x_data, y_data = create_dataset(dataset, look_back)

# 데이터를 RNN 입력에 맞게 변형
x_data = np.reshape(x_data, (x_data.shape[0], 1, x_data.shape[1]))

In [ ]:
model = Sequential()
model.add(SimpleRNN(100, activation='relu', input_shape=(1, look_back)))    # 50->100
model.add(Dense(50, activation='relu'))                                     # 25->50
model.add(Dense(1, activation='linear'))

model.summary()

In [ ]:
from keras.optimizers import Adam

model.compile(loss='mse', optimizer=Adam(learning_rate=0.001))                  # 학습률 0.001(0.0001 X)
model.fit(x_data, y_data, epochs=100, batch_size=32, verbose=1)                 # 배치 1->32

In [ ]:
# 향후 예측을 위한 입력 데이터 생성
x_test, y_test = create_dataset(dataset[look_back:], look_back)          # 수정 dataset -> dataset[look_back:]
x_test = np.reshape(x_test, (x_test.shape[0], 1, x_test.shape[1]))

predictions = model.predict(x_test)

In [ ]:
predictions

### **성능 평가**

밑밑이 원래 나와야하는 값인데 밑엔 다름

In [ ]:
print(f"x_data : {x_data.shape}")
print(f"y_data : {y_data.shape}")
print(f"x_test : {x_test.shape}")
print(f"y_test : {y_test.shape}")
print(f"predictions : {predictions.shape}")   # (8758) => rnn학습 -3(8755) => 예측 값 만들 때 -3(8752)

In [ ]:
print(f"x_data : {x_data.shape}")
print(f"y_data : {y_data.shape}")
print(f"x_test : {x_test.shape}")
print(f"y_test : {y_test.shape}")
print(f"predictions : {predictions.shape}")   # (8758) => rnn학습 -3(8755) => 예측 값 만들 때 -3(8752)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
mse = mean_squared_error(y_test[:len(predictions)], predictions)         # y_test[:len(predictions)] : 개수 맞추기
mae = mean_absolute_error(y_test[:len(predictions)], predictions)
print(f'MSE: {mse}')
print(f'MAE: {mae}')

### **정규화된 값 비교(실제 vs 예측)**

In [ ]:
plt.scatter(range(len(dataset)), dataset, label='Actual', color='blue')
plt.scatter(range(len(predictions)), predictions, label='Predicted', color='orange')

plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Values(Scale)')
plt.legend()
plt.show()

### **역정규화된 값 비교(실제 vs 예측)**

In [ ]:
dataset_rescale = scaler.inverse_transform(dataset)
pred_rescale = scaler.inverse_transform(predictions)

In [ ]:
pred_rescale

In [ ]:
result_df = pd.DataFrame({'Pred_Power Consumption(kWh)' : pred_rescale.flatten()})
df3['Pred_Power Consumption(kWh)'] = result_df['Pred_Power Consumption(kWh)']                   # 기존 df3에 예측한 거 추가

df3.to_csv('2018_RNN_pred_충북 충주 201801-201812.csv', index=False)

In [ ]:
print(dataset_rescale.shape)
print(pred_rescale.shape)

In [ ]:
## 밑엔 같은 거고 그래프 모양이 다른

In [ ]:
import matplotlib.pyplot as plt

plt.plot(dataset_rescale, label='True Data', color='blue')
plt.plot(range(look_back, len(predictions) + look_back), pred_rescale, label='Predictions', color='pink')

plt.title('Actual vs Predicted Values(rescale)')
plt.xlabel('Time')
plt.ylabel('Power Generation')
plt.legend()
plt.show()

In [ ]:
plt.scatter(range(len(dataset_rescale)), dataset_rescale, label='Actual', color='blue')
plt.scatter(range(look_back, len(predictions) + look_back), pred_rescale, label='Predicted', color='pink')
plt.xlabel('Index')
plt.ylabel('Value')
plt.title('Actual vs Predicted Values(rescale)')
plt.legend()
plt.show()

마지막

# **2018- ANN & RNN & Actual 비교**

In [ ]:
# 파-2018실제, 검-2018ann예측, 핑-2018rnn예측

In [ ]:
import matplotlib.pyplot as plt

plt.plot(dataset_rescale, label='Actual Data', color='blue')
plt.plot(y_pred_rescale2, label='ANN Predictions', color='black')
plt.plot(range(look_back, len(predictions) + look_back), pred_rescale, label='RNN Predictions', color='pink')

plt.title('2018-Solar Power Generation Prediction')
plt.xlabel('Time')
plt.ylabel('Power Generation')
plt.legend()
plt.show()

In [ ]:
plt.scatter(range(len(dataset_rescale)), dataset_rescale, label='Actual Data', color='blue')
plt.scatter(range(len(y_pred_rescale2)), y_pred_rescale2, label='ANN Predictions', color='black')
plt.scatter(range(look_back, len(predictions) + look_back), pred_rescale, label='RNN Predictions', color='pink')

plt.title('2018-Solar Power Generation Prediction')
plt.xlabel('Time')
plt.ylabel('Power Generation')
plt.legend()
plt.show()

In [ ]:
print(len(dataset_rescale))
print(len(y_pred_rescale2))
print(len(pred_rescale))

In [ ]:
print(range(len(dataset_rescale)))
print(range(len(y_pred_rescale2)))
print(range(look_back, len(predictions) + look_back))

##  2017 & 2018 실제값 비교 : 1시간 단위로 맞춘 데이터

In [ ]:
df_2017 = pd.read_csv("data/power_consumption_2017_hourly.csv", usecols=[6])
df_2017.dropna(inplace=True)

In [ ]:
df_2017

In [ ]:
y_rescale_sorted

In [ ]:
df_2017_array = df_2017['Zone 1 Power Consumption'].values.reshape(-1, 1)

In [ ]:
plt.scatter(range(len(y_rescale2)), y_rescale2, label='2018 data', color='pink')
plt.scatter(range(len(df_2017_array)), df_2017_array, label='2017 data', color='black')

plt.title('2017 & 2018 (actual)')
plt.xlabel('Time')
plt.ylabel('Power Generation')
plt.legend()
plt.show()

# 2018 자체 ANN vs RNN 예측값 비교

In [ ]:
plt.scatter(range(len(y_rescale_sorted)), y_rescale_sorted, label='2018 ANN', color='skyblue')   ## ..정렬 안되어있음..
plt.scatter(range(len(pred_rescale)), pred_rescale, label='2018 RNN', color='purple')

plt.title('2018- ANN vs RNN Pred')
plt.xlabel('Index')
plt.ylabel('Power Generation')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ['2017 ANN', '2018 ANN', '2018 RNN']
mae_values = [0.12852, 0.38820, 0.02253]
mse_values = [0.02558, 0.182706, 0.00134]

bar_width = 0.35
index = np.arange(len(models))

fig, ax = plt.subplots()
bar1 = ax.bar(index, mae_values, bar_width, label='MAE', color='green')
bar2 = ax.bar(index + bar_width, mse_values, bar_width, label='MSE', color='brown')

ax.set_xlabel('Models')
ax.set_ylabel('Errors')
ax.set_title('MAE & MSE')
ax.set_xticks(index + bar_width / 2)
ax.set_xticklabels(models)
ax.legend()

plt.show()